# Data Mesh on Databricks Lakehouse
## Mapping Concepts to Implementation

**3-Workspace Hub & Spoke architecture deployed via independent CI/CD pipelines**

**Repositories:**
* **Hub:** [github.com/santhosh-rajashekar/data-mesh-demo](https://github.com/santhosh-rajashekar/data-mesh-demo)
* **SNOW:** [github.com/santhosh-rajashekar/dbx-dp-snow](https://github.com/santhosh-rajashekar/dbx-dp-snow)
* **ADOIT:** [github.com/santhosh-rajashekar/dbx-dps-raise](https://github.com/santhosh-rajashekar/dbx-dps-raise)

**Reference Blogs:**
* [Databricks Lakehouse and Data Mesh, Part 1](https://www.databricks.com/blog/databricks-lakehouse-and-data-mesh-part-1)
* [Building Data Mesh Based on Databricks Lakehouse, Part 2](https://www.databricks.com/blog/building-data-mesh-based-databricks-lakehouse-part-2)
* [What is Data Mesh?](https://www.databricks.com/blog/what-is-data-mesh)

---
*This notebook maps Databricks Data Mesh blog concepts to a live, fully automated implementation deployed across 3 Azure Databricks workspaces via independent CI/CD pipelines.*

## What is Data Mesh? — The 4 Principles

> *"Data mesh is a paradigm that describes a set of principles and logical architecture for scaling data analytics platforms."* — Databricks Blog, Part 1

| Principle | Blog Definition | Our Demo Implementation |
| --- | --- | --- |
| **Domain Ownership** | Domain teams retain full responsibility for their data throughout its lifecycle | `adoit_product` catalog owned by EA Team, `snow_product` by ITSM Team — each domain owns full bronze → silver → gold |
| **Data as a Product** | Discoverable, trustworthy, self-describing, addressable, interoperable | 3 gold data products with tags, quality checks, SLA contracts, observability, registered in `data_product_registry` |
| **Self-Serve Platform** | Common tools and methods to build, run, and maintain data products | Databricks Asset Bundles + CI/CD (GitHub Actions), shared orchestrator job, reusable notebook patterns |
| **Federated Governance** | Central rules enforced through standardization | `data_mesh_hub` catalog with quality log, contracts, maturity scorecard, UDFs for column masking and row filtering |

## Hub & Spoke Topology — 3 Workspaces

> *"A Hub & Spoke Data Mesh incorporates a centralized location for managing shareable data assets and data that does not sit logically within any single domain."* — Databricks Blog, Part 2

Our demo implements **true workspace-per-domain isolation** — not just catalog separation, but physically separate Azure Databricks workspaces sharing a Unity Catalog metastore:

**Domain Spokes (Data Producers)**
* **ADOIT workspace** (`dbx-dps-raise-dev`): `https://adb-7405605730069993.13.azuredatabricks.net` → `adoit_product` catalog — Enterprise Architecture domain → produces **Application Landscape**
* **SNOW workspace** (`dbx-dps-snow-dev`): `https://adb-7405614225278648.8.azuredatabricks.net` → `snow_product` catalog — IT Service Management domain → produces **Service Health**

**Data Hub (Central Platform)**
* **Hub workspace** (`dbx-dps-hub`): `https://adb-7405616457611961.1.azuredatabricks.net` → `data_mesh_hub` catalog — **Application Risk Profile** (cross-domain join of EA + ITSM data)

**Key:** All 3 workspaces share one **Unity Catalog metastore**, enabling cross-catalog queries from the hub while maintaining true domain isolation. Each workspace has its own service principal, GitHub repo, and CI/CD pipeline.

In [0]:
%sql
-- Domain Ownership: 3 catalogs × 8 schemas — each domain owns its medallion tiers
SELECT catalog_name, schema_name, comment
FROM system.information_schema.schemata
WHERE catalog_name IN ('adoit_product', 'snow_product', 'data_mesh_hub')
  AND schema_name NOT IN ('information_schema', 'default')
  AND schema_name NOT LIKE 'dev_%'
ORDER BY catalog_name, schema_name

## Medallion Architecture per Domain

Each domain implements a full **bronze → silver → gold** pipeline independently in its own workspace:

| Tier | ADOIT (EA) — `dbx-dps-raise-dev` | SNOW (ITSM) — `dbx-dps-snow-dev` | Hub — `dbx-dps-hub` |
| --- | --- | --- | --- |
| **Bronze** | raw_applications, raw_capabilities, raw_mapping | raw_incidents, raw_change_requests | — |
| **Silver** | applications, business_capabilities | incidents, change_requests | — |
| **Gold** | application_landscape (14 cols) | service_health (12 cols) | application_risk_profile (22 cols) |
| **Governance** | registry, contracts, quality_log, observability | registry, contracts, quality_log, observability | registry, contracts, quality_log, observability, maturity_scorecard |

**Total: 25 tables across 3 catalogs, deployed independently via 3 GitHub repos**

In [0]:
%sql
-- Data Products: 3 gold-tier products across 2 domains + cross-domain hub
SELECT 'Application Landscape' AS data_product, 'Enterprise Architecture' AS domain, COUNT(*) AS records
FROM adoit_product.gold.application_landscape
UNION ALL
SELECT 'Service Health', 'IT Service Management', COUNT(*)
FROM snow_product.gold.service_health
UNION ALL
SELECT 'Application Risk Profile', 'Cross-Domain', COUNT(*)
FROM data_mesh_hub.cross_domain.application_risk_profile

Databricks visualization. Run in Databricks to view.

## Data Product Qualities

> *"Data products need to be discoverable, trustworthy, self-describing, addressable and interoperable."* — Databricks Blog, Part 1

| Quality | Blog Definition | Our Implementation |
| --- | --- | --- |
| **Discoverable** | Easy to find via catalog | Unity Catalog tags (`data_product=true`, `domain`, `pii`, `classification`) + `data_product_registry` |
| **Trustworthy** | High quality, reliable | 8 automated quality checks, all passing, logged in `data_quality_log` |
| **Self-describing** | Documentation built in | Table and column comments, `schema_version` in contracts |
| **Addressable** | Clear location | Fully qualified names: `adoit_product.gold.application_landscape` |
| **Interoperable** | Works across domains | `application_risk_profile` joins EA + ITSM data seamlessly |

In [0]:
%sql
-- Discoverability: Unity Catalog tags on all gold data product tables
SELECT catalog_name, schema_name, table_name, tag_name, tag_value
FROM system.information_schema.table_tags
WHERE tag_name = 'data_product'
ORDER BY catalog_name

In [0]:
%sql
-- Trustworthiness: 8 automated quality checks across all data products
SELECT data_product, check_name, check_type,
       CASE WHEN passed THEN '✅ PASS' ELSE '❌ FAIL' END AS status,
       expected_value, actual_value, severity
FROM data_mesh_hub.platform.data_quality_log
ORDER BY data_product, check_name

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Interoperability: Cross-domain join — Risk Profile combining EA + ITSM data
SELECT rp.app_name, rp.criticality, rp.composite_risk_score,
       rp.sla_compliance_pct, rp.total_incidents, rp.total_changes,
       al.department, al.lifecycle_status
FROM data_mesh_hub.cross_domain.application_risk_profile rp
JOIN adoit_product.gold.application_landscape al ON rp.app_id = al.app_id
ORDER BY rp.composite_risk_score DESC

Databricks visualization. Run in Databricks to view.

## Federated Governance

> *"Federated governance ensures central, consistent data governance across domains. Compliance is tracked and managed centrally via a data catalog, data governance tools and automated policy enforcement."* — Databricks Blog, "What is Data Mesh"

Our governance stack in `data_mesh_hub.platform`:

* **Data Contracts** — 5 active contracts with SLA hours, quality thresholds, escalation contacts per consumer
* **Quality Monitoring** — Automated checks for completeness, uniqueness, validity, referential integrity
* **Product Observability** — Row counts, freshness tracking, quality scores, schema drift detection
* **Domain Maturity Scorecard** — Quantified scores across quality, accessibility, documentation, security
* **Row-Level Security** — `filter_by_risk` UDF: risk committee sees all apps, others see only low/medium risk
* **Column Masking** — `mask_contact` UDF: non-admins see `ea-***@***.com` instead of real emails
* **Change Data Feed** — Enabled on all gold tables for incremental consumption

In [0]:
%sql
-- Federated Governance: Data contracts define SLAs between producers and consumers
SELECT contract_id, product_name, producer_group, consumer_group,
       contract_status, agreed_sla_hours, quality_threshold_pct, escalation_contact
FROM data_mesh_hub.platform.data_contracts
ORDER BY contract_id

In [0]:
%sql
-- Domain Maturity: Quantified governance scores per domain
SELECT domain, maturity_level, overall_maturity_score,
       data_quality_score, accessibility_score, documentation_score,
       interoperability_score, governance_score, security_score,
       observability_score, self_serve_score, product_count, consumer_count, uptime_pct
FROM data_mesh_hub.platform.domain_maturity_scorecard
ORDER BY overall_maturity_score DESC

Databricks visualization. Run in Databricks to view.

## Databricks Capabilities Mapping

> *"Several of Databricks' largest customers worldwide have adopted data mesh using the Lakehouse as the technological foundation."* — Databricks Blog, Part 1

| Databricks Capability | Data Mesh Role | Our Demo |
| --- | --- | --- |
| **Unity Catalog** | Federated governance, discovery, lineage | 3 catalogs, 10+ schemas, 25 tables, tags, column masking, row filters |
| **Workspace-per-Domain** | True domain isolation | 3 Azure Databricks workspaces with shared Unity Catalog metastore |
| **Medallion Architecture** | Organize data within domains | bronze → silver → gold per domain workspace |
| **Delta Sharing** | Share data products across organizations | 3 shares for gold data products |
| **Catalog-per-domain** | Domain ownership boundary | `adoit_product`, `snow_product`, `data_mesh_hub` |
| **Asset Bundles** | Self-service infrastructure platform | 3 separate `databricks.yml` configs, one per workspace |
| **Service Principals** | Domain-scoped CI/CD identity | 3 SPs: `data-mesh-cicd`, `snow-domain-cicd`, `raise-domain-cicd` |
| **Tags** | Data product discoverability | `data_product`, `domain`, `pii`, `classification` |
| **Change Data Feed** | Incremental data consumption | Enabled on all gold tables |

## CI/CD — Self-Service Platform

> *"Taking a domain-agnostic approach to the data analytics lifecycle, using common tools and methods to build, run, and maintain interoperable data products."* — Databricks Blog, Part 1

**3 GitHub Repositories — Independent CI/CD per Domain:**

| Repo | Workspace | Service Principal | Pipeline |
| --- | --- | --- | --- |
| [data-mesh-demo](https://github.com/santhosh-rajashekar/data-mesh-demo) | Hub (`dbx-dps-hub`) | `data-mesh-cicd` | 6 tasks: setup → cross_domain → quality → governance → advanced → exchange |
| [dbx-dp-snow](https://github.com/santhosh-rajashekar/dbx-dp-snow) | SNOW (`dbx-dps-snow-dev`) | `snow-domain-cicd` | 3 tasks: setup → pipeline → governance |
| [dbx-dps-raise](https://github.com/santhosh-rajashekar/dbx-dps-raise) | ADOIT (`dbx-dps-raise-dev`) | `raise-domain-cicd` | 3 tasks: setup → pipeline → governance |

**Each Repo Has:**
* `validate.yml` — YAML parsing, schema checks, column matching
* `deploy-dev.yml` — Bundle deploy + pipeline run (triggered on push or workflow_dispatch)
* `databricks.yml` — Infrastructure as code, workspace-specific

**Key Design Choices:**
* Each domain is **fully autonomous** — can deploy independently without touching other domains
* Service principals are **domain-scoped** — SNOW SP can only write to `snow_product`, ADOIT SP only to `adoit_product`
* Hub reads from domain catalogs via **cross-catalog grants** — no data copying

In [0]:
%sql
-- Self-Serve Observability: automated health monitoring for each data product
SELECT product_name, quality_score, freshness_hours,
       CASE WHEN sla_met THEN '✅ Met' ELSE '❌ Breached' END AS sla_status,
       row_count, consumer_count, status
FROM data_mesh_hub.platform.product_observability
ORDER BY product_name

Databricks visualization. Run in Databricks to view.

## Delta Sharing — Data as a Product Beyond Boundaries

> *"Delta Sharing offers a solution to securely share data products between domains across organizational, regional, and technology boundaries."* — Databricks Blog, Part 2

**3 Delta Shares Created:**
* `adoit_application_landscape` → Application Landscape (EA Domain)
* `snow_service_health` → Service Health (ITSM Domain)
* `datamesh_application_risk_profile` → Application Risk Profile (Cross-Domain)

**Why Delta Sharing for Data Mesh:**
* **Open protocol** — consumers don't need Databricks
* **No data copying** — consumers query live, governed data
* **Unity Catalog governed** — access controls enforced at the share level
* **Cross-organizational** — share data products with external partners, regulators, or other business units

## Deviations & Simplifications for Demo

| Blog Recommends | Demo Status | Details |
| --- | --- | --- |
| Separate workspace per domain | ✅ **IMPLEMENTED** | 3 Azure Databricks workspaces with shared Unity Catalog metastore |
| Domain teams manage own pipelines | ✅ **IMPLEMENTED** | Each domain has own GitHub repo, service principal, and CI/CD pipeline |
| Streaming data products | Batch (CSV → MERGE) | Production: Auto Loader + Structured Streaming |
| Marketplace listings for sharing | Delta Sharing shares only | Production: Full Marketplace provider + private exchange |
| Terraform for infrastructure | Databricks Asset Bundles | Either works — Bundles are Databricks-native |

*The first two blog recommendations — workspace-per-domain and independent pipelines — are now fully implemented. The remaining simplifications keep the demo focused while each has a clear production upgrade path.*

## Summary

| Data Mesh Principle | Status | Implementation |
| --- | --- | --- |
| Domain Ownership | ✅ | 3 workspaces, catalog-per-domain, independent CI/CD per domain |
| Data as a Product | ✅ | Discoverable, trustworthy, self-describing, addressable, interoperable |
| Self-Serve Platform | ✅ | 3 GitHub repos + Asset Bundles + 3 service principals |
| Federated Governance | ✅ | Central quality, contracts, observability, security — plus domain-level governance |
| Hub & Spoke Topology | ✅ | 3 workspaces: 2 domain spokes + 1 central hub, shared metastore |
| Delta Sharing | ✅ | Cross-organization data product distribution |

**Architecture: 25 tables across 3 catalogs, 3 workspaces, 3 repos, 3 service principals**

---

> *"Everything you see — catalogs, tables, quality checks, dashboard, sharing — is defined in code and deployed through independent CI/CD pipelines across 3 workspaces."*

**GitHub Repos:**
* [data-mesh-demo](https://github.com/santhosh-rajashekar/data-mesh-demo) — Hub (cross-domain, governance, dashboard)
* [dbx-dp-snow](https://github.com/santhosh-rajashekar/dbx-dp-snow) — SNOW domain (ITSM)
* [dbx-dps-raise](https://github.com/santhosh-rajashekar/dbx-dps-raise) — ADOIT domain (EA)